In [1]:
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt

# 1. Configuration
BATCH_SIZE = 32
IMAGE_SIZE = (256, 256)
CHANNELS = 3
EPOCHS = 5 

# 2. Load Data with an 80/20 Train/Validation Split
print("Loading Training Data...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/raw",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

print("Loading Validation Data...")
val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/raw",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names

# 3. Optimize dataset for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 4. Build the CNN Model
resize_and_rescale = tf.keras.Sequential([
  layers.Resizing(256, 256),
  layers.Rescaling(1./255), # Normalize pixel values to 0-1
])

model = models.Sequential([
    resize_and_rescale,
    layers.Conv2D(32, kernel_size=(3,3), activation='relu', input_shape=(BATCH_SIZE, 256, 256, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64,  kernel_size=(3,3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64,  kernel_size=(3,3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(class_names), activation='softmax') # Output layer matches number of diseases
])

# 5. Compile the Model
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# 6. Train the Model
print("Starting Training...")
history = model.fit(
    train_ds,
    batch_size=BATCH_SIZE,
    validation_data=val_ds,
    verbose=1,
    epochs=EPOCHS,
)

# 7. Save the Model for the Web-GUI
model.save("plant_disease_model.keras")
print("Model saved successfully as plant_disease_model.keras!")

Loading Training Data...
Found 54306 files belonging to 38 classes.
Using 43445 files for training.
Loading Validation Data...
Found 54306 files belonging to 38 classes.
Using 10861 files for validation.
Starting Training...
Epoch 1/5
1358/1358 [==============================] - 1005s 719ms/step - loss: 1.1962 - accuracy: 0.6621 - val_loss: 0.6219 - val_accuracy: 0.8136
Epoch 2/5
1358/1358 [==============================] - 992s 730ms/step - loss: 0.3857 - accuracy: 0.8792 - val_loss: 0.4540 - val_accuracy: 0.8606
Epoch 3/5
1358/1358 [==============================] - 1653s 1s/step - loss: 0.1877 - accuracy: 0.9398 - val_loss: 0.4913 - val_accuracy: 0.8613
Epoch 4/5
1358/1358 [==============================] - 4631s 3s/step - loss: 0.0970 - accuracy: 0.9688 - val_loss: 0.5422 - val_accuracy: 0.8697
Epoch 5/5
1358/1358 [==============================] - 1015s 747ms/step - loss: 0.0820 - accuracy: 0.9732 - val_loss: 0.5345 - val_accuracy: 0.8751
Model saved successfully as plant_disease_